In [ ]:
import tkinter as tk
import random
import time
from collections import deque

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
TILE = 90
GAP = 5
ANIM = 300

def misplaced(board):
    return sum(1 for i, val in enumerate(board) if val != 0 and val != GOAL[i])

def apply_action(board, action):
    idx = board.index(0)
    r, c = divmod(idx, 3)
    dr, dc = action
    nr, nc = r + dr, c + dc
    if 0 <= nr < 3 and 0 <= nc < 3:
        ni = nr * 3 + nc
        temp = list(board)
        temp[idx], temp[ni] = temp[ni], temp[idx]
        return tuple(temp)
    return None

ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_NAMES = {(-1, 0): "UP", (1, 0): "DOWN", (0, -1): "LEFT", (0, 1): "RIGHT"}

def sensorless_bfs(board1, board2):
    start_belief = frozenset([board1, board2])
    goal_belief = frozenset([GOAL])

    if start_belief == goal_belief:
        return [], [], 0
    queue = deque()
    queue.append((start_belief, [], [start_belief]))
    visited = {start_belief}
    nodes = 0

    while queue:
        belief, actions, belief_path = queue.popleft()
        nodes += 1

        for action in ACTIONS:
            new_belief_set = set()
            for board in belief:
                result = apply_action(board, action)
                if result is not None:
                    new_belief_set.add(result)
                else:
                    new_belief_set.add(board) 

            new_belief = frozenset(new_belief_set)
            if new_belief in visited:
                continue
            visited.add(new_belief)
            new_actions = actions + [action]
            new_path = belief_path + [new_belief]

            if new_belief == goal_belief or all(b == GOAL for b in new_belief):
                return new_actions, new_path, nodes
            queue.append((new_belief, new_actions, new_path))
            if nodes > 200000:
                return new_actions, new_path, nodes

    return [], [], nodes


def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = sum(
        1 for i in range(len(arr))
        for j in range(i + 1, len(arr))
        if arr[i] > arr[j]
    )
    return inv % 2 == 0

def random_board():
    b = list(GOAL)
    while True:
        random.shuffle(b)
        t = tuple(b)
        if is_solvable(t) and t != GOAL:
            return t


class App:
    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle")
        self.root.configure(bg="white")
        self.root.resizable(False, False)

        self.board1 = random_board()
        self.board2 = random_board()
        while self.board2 == self.board1:
            self.board2 = random_board()

        self.solution_actions = []
        self.belief_path = []
        self.step = 0
        self.playing = False

        main = tk.Frame(root, bg="white")
        main.pack(padx=10, pady=10)

        left = tk.Frame(main, bg="white")
        left.pack(side="left", padx=10)
        boards_frame = tk.Frame(left, bg="white")
        boards_frame.pack()

        b1_frame = tk.Frame(boards_frame, bg="white")
        b1_frame.pack(side="left", padx=6)
        tk.Label(b1_frame, text="Board 1", font=("Arial", 11, "bold"),
                 bg="white", fg="#0055aa").pack()
        self.canvas1 = tk.Canvas(
            b1_frame,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white", highlightthickness=0
        )
        self.canvas1.pack()
        self.lb_h1 = tk.Label(b1_frame, text="h(n) = —",
                               bg="white", fg="#c05000", font=("Arial", 10))
        self.lb_h1.pack(pady=2)

        b2_frame = tk.Frame(boards_frame, bg="white")
        b2_frame.pack(side="left", padx=6)
        tk.Label(b2_frame, text="Board 2", font=("Arial", 11, "bold"),
                 bg="white", fg="#aa5500").pack()
        self.canvas2 = tk.Canvas(
            b2_frame,
            width=3 * TILE + 4 * GAP,
            height=3 * TILE + 4 * GAP,
            bg="white", highlightthickness=0
        )
        self.canvas2.pack()
        self.lb_h2 = tk.Label(b2_frame, text="h(n) = —",
                               bg="white", fg="#c05000", font=("Arial", 10))
        self.lb_h2.pack(pady=2)

        self.step_label = tk.Label(
            left, text="Steps:", font=("Arial", 12), bg="white"
        )
        self.step_label.pack(pady=4)

        info = tk.Frame(left, bg="white")
        info.pack(pady=4)
        self.lb_nodes = tk.Label(info, text="Nodes: 0", bg="white")
        self.lb_nodes.grid(row=0, column=0, padx=6)
        self.lb_steps = tk.Label(info, text="Steps: 0", bg="white")
        self.lb_steps.grid(row=0, column=1, padx=6)
        self.lb_time = tk.Label(info, text="Time: 0 ms", bg="white")
        self.lb_time.grid(row=0, column=2, padx=6)

        info2 = tk.Frame(left, bg="white")
        info2.pack(pady=2)
        self.lb_belief = tk.Label(
            info2, text="Belief size: 2", bg="white", fg="#006400", font=("Arial", 10)
        )
        self.lb_belief.grid(row=0, column=0, padx=6)
        self.lb_visited = tk.Label(
            info2, text="Visited: 0", bg="white", fg="#00008b", font=("Arial", 10)
        )
        self.lb_visited.grid(row=0, column=1, padx=6)

        btns = tk.Frame(left, bg="white")
        btns.pack(pady=6)
        tk.Button(btns, text="Shuffle", width=10, command=self.shuffle).grid(row=0, column=0, padx=4)
        tk.Button(btns, text="Reset",   width=10, command=self.reset  ).grid(row=0, column=1, padx=4)
        tk.Button(btns, text="Solve",   width=10, command=self.solve  ).grid(row=0, column=2, padx=4)
        tk.Button(btns, text="Step",    width=34, command=self.next_step).grid(
            row=1, column=0, columnspan=3, pady=4
        )

        right = tk.Frame(main, bg="white")
        right.pack(side="right", padx=10)

        tk.Label(
            right,
            text="Solution (Sensorless BFS)",
            font=("Arial", 13, "bold"),
            bg="white"
        ).pack()

        self.solution_box = tk.Text(right, width=26, height=25, font=("Consolas", 10))
        self.solution_box.pack()

        self.draw()

    def get_boards_at_step(self, step):
        if not self.belief_path:
            return self.board1, self.board2
        belief = self.belief_path[step]
        boards = list(belief)
        if len(boards) == 1:
            return boards[0], boards[0]
        return boards[0], boards[1]

    def draw_board(self, canvas, board, label_widget):
        canvas.delete("all")
        h_val = misplaced(board)
        for i, val in enumerate(board):
            r, c = divmod(i, 3)
            x1 = GAP + c * (TILE + GAP)
            y1 = GAP + r * (TILE + GAP)
            x2 = x1 + TILE
            y2 = y1 + TILE
            if val == 0:
                canvas.create_rectangle(
                    x1, y1, x2, y2, fill="#e8f5e9", outline="#aaa",
                    width=2, dash=(4, 3)
                )
            else:
                wrong = (val != GOAL[i])
                fill = "#ffe0e0" if wrong else "white"
                canvas.create_rectangle(
                    x1, y1, x2, y2, fill=fill, outline="black", width=2
                )
                canvas.create_text(
                    (x1 + x2) // 2, (y1 + y2) // 2,
                    text=str(val), font=("Arial", 28),
                    fill="red" if wrong else "black"
                )
        label_widget.config(text=f"h(n) = {h_val}")

    def draw(self):
        if self.belief_path and self.step < len(self.belief_path):
            b1, b2 = self.get_boards_at_step(self.step)
        else:
            b1, b2 = self.board1, self.board2
        self.draw_board(self.canvas1, b1, self.lb_h1)
        self.draw_board(self.canvas2, b2, self.lb_h2)
        if self.belief_path and self.step < len(self.belief_path):
            self.lb_belief.config(text=f"Belief size: {len(self.belief_path[self.step])}")

    def shuffle(self):
        self.playing = False
        self.board1 = random_board()
        self.board2 = random_board()
        while self.board2 == self.board1:
            self.board2 = random_board()
        self.solution_actions = []
        self.belief_path = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def reset(self):
        self.playing = False
        self.board1 = GOAL
        self.board2 = GOAL
        self.solution_actions = []
        self.belief_path = []
        self.step = 0
        self._reset_labels()
        self.draw()

    def _reset_labels(self):
        self.lb_nodes.config(text="Nodes: 0")
        self.lb_steps.config(text="Steps: 0")
        self.lb_time.config(text="Time: 0 ms")
        self.step_label.config(text="Steps:")
        self.lb_h1.config(text="h(n) = —")
        self.lb_h2.config(text="h(n) = —")
        self.lb_belief.config(text="Belief size: 2")
        self.lb_visited.config(text="Visited: 0")
        self.solution_box.delete("1.0", tk.END)

    def solve(self):
        if self.playing:
            return
        t0 = time.perf_counter()
        actions, belief_path, nodes = sensorless_bfs(self.board1, self.board2)
        ms = (time.perf_counter() - t0) * 1000

        self.solution_actions = actions
        self.belief_path = belief_path
        self.step = 0

        self.lb_nodes.config(text=f"Nodes: {nodes}")
        self.lb_steps.config(text=f"Steps: {len(actions)}")
        self.lb_time.config(text=f"Time: {ms:.2f} ms")
        self.lb_visited.config(text=f"Visited: {nodes}")

        if belief_path:
            final_belief = belief_path[-1]
            solved = all(b == GOAL for b in final_belief)
            self.step_label.config(text="✓ SOLVED" if solved else "✗ Not solved")
        else:
            self.step_label.config(text="✗ No solution")

        self.solution_box.delete("1.0", tk.END)
        if actions:
            self.playing = True
            self.animate()

    def _update_solution_box(self):
        if not self.belief_path or self.step == 0:
            return
        self.solution_box.config(state=tk.NORMAL)
        self.solution_box.delete("1.0", tk.END)
        for i in range(1, self.step + 1):
            act = self.solution_actions[i - 1]
            name = ACTION_NAMES[act]
            belief = self.belief_path[i]
            h_vals = [misplaced(b) for b in belief]
            h_str = "/".join(str(h) for h in sorted(h_vals))
            self.solution_box.insert(tk.END, f"{i:2}. {name:<5} h={h_str}\n")
        self.solution_box.tag_remove("highlight", "1.0", tk.END)
        self.solution_box.tag_add("highlight", f"{self.step}.0", f"{self.step}.end")
        self.solution_box.tag_config("highlight", background="#cce5ff")
        self.solution_box.see(tk.END)

    def animate(self):
        if not self.playing:
            return
        self.draw()
        if self.step == 0:
            self.step_label.config(text="START")
        else:
            act = self.solution_actions[self.step - 1]
            self.step_label.config(text=f"Step {self.step}: {ACTION_NAMES[act]}")

        self._update_solution_box()

        if self.step < len(self.belief_path) - 1:
            self.step += 1
            self.root.after(ANIM, self.animate)
        else:
            self.playing = False
            final_belief = self.belief_path[-1]
            solved = all(b == GOAL for b in final_belief)
            self.step_label.config(text="✓ SOLVED!" if solved else "✗ Stuck")

    def next_step(self):
        self.playing = False
        if not self.belief_path:
            return
        if self.step < len(self.belief_path) - 1:
            self.step += 1
            self.draw()
            act = self.solution_actions[self.step - 1]
            self.step_label.config(text=f"Step {self.step}: {ACTION_NAMES[act]}")
            self._update_solution_box()


if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()